# **Latest Code**

In [2]:
import os

repo_name = "TIMEGAN-SyntheticData-FinancialFraudDetection"
target_dir = f"/content/{repo_name}"

# Check if repo exists on disk
if os.path.exists(target_dir):
    %cd {target_dir}
    print(f"Switched directory to: {os.getcwd()}")
    !git pull origin main
else:
    # If runtime was reset or repo was deleted, re-clone
    %cd /content
    print(f"Repository not found at {target_dir}. Re-cloning...")
    !git clone https://github.com/syaahiddd/TIMEGAN-SyntheticData-FinancialFraudDetection.git
    %cd {target_dir}
    print(f"Switched directory to: {os.getcwd()}")

# Verify git tracking
!git status -s

/content
Repository not found at /content/TIMEGAN-SyntheticData-FinancialFraudDetection. Re-cloning...
Cloning into 'TIMEGAN-SyntheticData-FinancialFraudDetection'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 52 (delta 21), reused 32 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 27.37 KiB | 9.12 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/TIMEGAN-SyntheticData-FinancialFraudDetection
Switched directory to: /content/TIMEGAN-SyntheticData-FinancialFraudDetection


In [3]:
import sys
import importlib
import src.timegan

# Reload module in case Python cached the old version in runtime memory
importlib.reload(src.timegan)

from src.timegan import TimeGAN, get_device

device = get_device()
print(f"Device: {device}")
print(f"Embedding Activation: {src.timegan.EmbeddingNetwork(10, 32).activation}")
print(f"Discriminator Activation: {src.timegan.DiscriminatorNetwork(32).leaky_relu}")

Device: cpu
Embedding Activation: Tanh()
Discriminator Activation: LeakyReLU(negative_slope=0.2)


In [9]:
import os
import torch
from google.colab import drive

# Re-affirm Drive directory
DRIVE_DIR = '/content/drive/MyDrive/TimeGAN_FYP_Outputs'
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/synthetic", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/figures", exist_ok=True)

N_SAMPLES, SEQ_LEN, D_FEATURES = real_train_x.shape

model = TimeGAN(
    feature_dim=D_FEATURES,
    seq_len=SEQ_LEN,
    hidden_dim=32,      # 32 units to capture multi-feature dynamics
    num_layers=3,
    lr=1e-3,
    gamma=1.0,
    device=device
)

# Recommended epochs for reliable convergence without excessive runtime:
# Phase 1 & 2: 30 epochs | Phase 3: 60 epochs (~10-15 minutes on T4 GPU)
history = model.fit(
    real_train_x,
    batch_size=128,
    pretrain_epochs=30,
    joint_epochs=60
)

# Save updated weights
torch.save(model.generator.state_dict(), f"{DRIVE_DIR}/checkpoints/generator_v2.pt")
torch.save(model.recovery.state_dict(), f"{DRIVE_DIR}/checkpoints/recovery_v2.pt")
torch.save(model.embedder.state_dict(), f"{DRIVE_DIR}/checkpoints/embedder_v2.pt")
torch.save(model.supervisor.state_dict(), f"{DRIVE_DIR}/checkpoints/supervisor_v2.pt")

print("\nModel trained and updated checkpoints saved.")

NameError: name 'real_train_x' is not defined

In [11]:
# 2. Mount Google Drive safely (only if not already mounted)
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")

DRIVE_DIR = '/content/drive/MyDrive/TimeGAN_FYP_Outputs'
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/synthetic", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/figures", exist_ok=True)

Google Drive is already mounted.


In [12]:
import os
import sys
import numpy as np
import pandas as pd
import torch
from google.colab import drive

# 1. Ensure current directory is in sys.path
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.data_loader import FinancialTimeSeriesLoader
from src.timegan import TimeGAN, get_device

# 2. Mount Google Drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/TimeGAN_FYP_Outputs'
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/synthetic", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/figures", exist_ok=True)

# 3. Load & Preprocess Multi-Feature Data
DATA_PATH = "data/raw/creditcard.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "/content/data/raw/creditcard.csv"

print(f"Loading data from {DATA_PATH}...")
df = pd.read_csv(DATA_PATH)

# Assign names if headers are missing
if "Class" not in df.columns:
    n_features = df.shape[1] - 2
    df.columns = ["Time"] + [f"V{i}" for i in range(1, n_features + 1)] + ["Class"]

df["Class"] = pd.to_numeric(df["Class"], errors="coerce").fillna(0).astype(int)

# Log-transform skewed Amount and Time deltas
if "Amount" in df.columns:
    df["Amount"] = np.log1p(df["Amount"].clip(lower=0))

if "Time" in df.columns:
    df = df.sort_values(by="Time").reset_index(drop=True)
    df["delta_time"] = np.log1p(df["Time"].diff().fillna(0.0).clip(lower=0))

# Select all features except raw Time and Class
exclude_cols = {"Time", "Class"}
modeled_features = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]

SEQ_LEN = 24
loader = FinancialTimeSeriesLoader(seq_len=SEQ_LEN, scaler_range=(0.0, 1.0))
scaled_matrix = loader.fit_transform_tabular(df, feature_cols=modeled_features)
sequences = loader.build_sliding_windows(scaled_matrix, stride=1)

# Sequence labels
targets = df["Class"].values
window_labels = []
for i in range(0, len(targets) - SEQ_LEN + 1, 1):
    window_labels.append(int(np.any(targets[i : i + SEQ_LEN] == 1)))
window_labels = np.array(window_labels, dtype=int)

# 80/20 train/test split
split_idx = int(len(sequences) * 0.8)
real_train_x, real_test_x = sequences[:split_idx], sequences[split_idx:]
real_train_y, real_test_y = window_labels[:split_idx], window_labels[split_idx:]

N_SAMPLES, SEQ_LEN, D_FEATURES = real_train_x.shape
print(f"-> Tensor verified: {real_train_x.shape} (N={N_SAMPLES}, T={SEQ_LEN}, D={D_FEATURES})")

# 4. Initialize TimeGAN on GPU
device = get_device()
print(f"-> Compute engine: {device}")

model = TimeGAN(
    feature_dim=D_FEATURES,
    seq_len=SEQ_LEN,
    hidden_dim=32,
    num_layers=3,
    lr=1e-3,
    gamma=1.0,
    device=device
)

# 5. Launch Retraining
BATCH_SIZE = 128
PRETRAIN_EPOCHS = 30
JOINT_EPOCHS = 60

print("=" * 60)
print(f"Training TimeGAN: {PRETRAIN_EPOCHS} Pretrain / {JOINT_EPOCHS} Joint Epochs")
print("=" * 60)

history = model.fit(
    real_train_x,
    batch_size=BATCH_SIZE,
    pretrain_epochs=PRETRAIN_EPOCHS,
    joint_epochs=JOINT_EPOCHS
)

# 6. Save Model Checkpoints
torch.save(model.generator.state_dict(), f"{DRIVE_DIR}/checkpoints/generator_v2.pt")
torch.save(model.recovery.state_dict(), f"{DRIVE_DIR}/checkpoints/recovery_v2.pt")
torch.save(model.embedder.state_dict(), f"{DRIVE_DIR}/checkpoints/embedder_v2.pt")
torch.save(model.supervisor.state_dict(), f"{DRIVE_DIR}/checkpoints/supervisor_v2.pt")
print(f"\nModel checkpoints successfully saved to {DRIVE_DIR}/checkpoints/")

ValueError: Mountpoint must not already contain files

In [13]:
import os
import sys
import numpy as np
import pandas as pd
import torch
from google.colab import drive

# 1. Ensure project root is discoverable
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.data_loader import FinancialTimeSeriesLoader
from src.timegan import TimeGAN, get_device

# 2. Mount Google Drive safely (bypasses if already mounted)
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")

DRIVE_DIR = '/content/drive/MyDrive/TimeGAN_FYP_Outputs'
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/synthetic", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/figures", exist_ok=True)

# 3. Ingest and preprocess tabular data
DATA_PATH = "/content/data/raw/creditcard.csv"
SEQ_LEN = 24

df = pd.read_csv(DATA_PATH)

# Standardize columns if raw CSV had no headers
if "Class" not in df.columns:
    n_features = df.shape[1] - 2
    df.columns = ["Time"] + [f"V{i}" for i in range(1, n_features + 1)] + ["Class"]

df["Class"] = pd.to_numeric(df["Class"], errors="coerce").fillna(0).astype(int)

# Log-transform skewed Amount feature
if "Amount" in df.columns:
    df["Amount"] = np.log1p(df["Amount"].clip(lower=0))

# Compute and log-transform delta_time
if "Time" in df.columns:
    df = df.sort_values(by="Time").reset_index(drop=True)
    df["delta_time"] = np.log1p(df["Time"].diff().fillna(0.0).clip(lower=0))

# Select all numerical features except Time and Class
exclude_cols = {"Time", "Class"}
modeled_features = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
print(f"Total Modeled Features: {len(modeled_features)}")

# Build sliding-window tensors
loader = FinancialTimeSeriesLoader(seq_len=SEQ_LEN, scaler_range=(0.0, 1.0))
scaled_matrix = loader.fit_transform_tabular(df, feature_cols=modeled_features)
sequences = loader.build_sliding_windows(scaled_matrix, stride=1)

# Generate window labels (1 if fraud exists anywhere in sequence window)
targets = df["Class"].values
window_labels = []
for i in range(0, len(targets) - SEQ_LEN + 1, 1):
    window_labels.append(int(np.any(targets[i : i + SEQ_LEN] == 1)))
window_labels = np.array(window_labels, dtype=int)

# 80/20 train/test split
split_idx = int(len(sequences) * 0.8)
real_train_x, real_test_x = sequences[:split_idx], sequences[split_idx:]
real_train_y, real_test_y = window_labels[:split_idx], window_labels[split_idx:]

N_SAMPLES, SEQ_LEN, D_FEATURES = real_train_x.shape
print(f"Data verified: real_train_x shape = {real_train_x.shape} (N={N_SAMPLES}, T={SEQ_LEN}, D={D_FEATURES})")

# 4. Initialize TimeGAN on GPU
device = get_device()
print(f"Launching TimeGAN on device: {device}")

model = TimeGAN(
    feature_dim=D_FEATURES,
    seq_len=SEQ_LEN,
    hidden_dim=32,
    num_layers=3,
    lr=1e-3,
    gamma=1.0,
    device=device
)

# 5. Execute 3-phase training
BATCH_SIZE = 128
PRETRAIN_EPOCHS = 30
JOINT_EPOCHS = 60

print("=" * 65)
print(f"Starting Multi-Phase Training: Batch Size={BATCH_SIZE}")
print(f"Phases 1 & 2: {PRETRAIN_EPOCHS} epochs each | Phase 3: {JOINT_EPOCHS} epochs")
print("=" * 65)

history = model.fit(
    real_train_x,
    batch_size=BATCH_SIZE,
    pretrain_epochs=PRETRAIN_EPOCHS,
    joint_epochs=JOINT_EPOCHS
)

# 6. Save model checkpoints to Google Drive
torch.save(model.generator.state_dict(), f"{DRIVE_DIR}/checkpoints/generator_v2.pt")
torch.save(model.recovery.state_dict(), f"{DRIVE_DIR}/checkpoints/recovery_v2.pt")
torch.save(model.embedder.state_dict(), f"{DRIVE_DIR}/checkpoints/embedder_v2.pt")
torch.save(model.supervisor.state_dict(), f"{DRIVE_DIR}/checkpoints/supervisor_v2.pt")

print("\n" + "=" * 65)
print("Training completed successfully.")
print(f"Checkpoints saved to: {DRIVE_DIR}/checkpoints/")
print("=" * 65)

Google Drive is already mounted.


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/raw/creditcard.csv'

# TimeGAN Financial Fraud Detection - Training Pipeline (Google Colab)
This notebook connects to your GitHub repository, mounts Google Drive for persistent checkpoints, and handles GPU-accelerated training.

### 1. Mount Google Drive
Persistent storage for model checkpoints (`.pt`), synthetic outputs, and evaluation metrics across Colab session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Create an external checkpoint backup folder in your Google Drive
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/FYP_TimeGAN_Checkpoints'
os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
print(f"Persistent Drive directory ready at: {DRIVE_PROJECT_PATH}")

### 2. Verify GPU Hardware Acceleration

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
cuda_available = torch.cuda.is_available()
print("CUDA Available:", cuda_available)

if cuda_available:
    print("Active GPU:", torch.cuda.get_device_name(0))
    print("Total Memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> Select T4 GPU.")

### 3. Clone / Pull Repository & Install Dependencies
For private repositories, use a GitHub Personal Access Token (PAT).

In [ ]:
# Set your GitHub credentials here
GITHUB_USERNAME = "<your-username>"
GITHUB_TOKEN = "<your-personal-access-token>"  # Leave blank if repository is Public
REPO_NAME = "TIMEGAN-SyntheticData-FinancialFraudDetection"

import os
if not os.path.exists(REPO_NAME):
    if GITHUB_TOKEN and GITHUB_TOKEN != "<your-personal-access-token>":
        repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
    else:
        repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
    !git clone {repo_url}
else:
    print("Repository already exists. Pulling latest commits...")
    %cd {REPO_NAME}
    !git pull
    %cd ..

# Navigate into repo and install packages
%cd {REPO_NAME}
!pip install -r requirements.txt -q
print("Dependencies installed successfully.")

### 4. Test Modular Import
Ensure `src` modules are recognized by Colab's Python path.

In [ ]:
import sys
sys.path.append('.')

# Verify that data_loader imports correctly
from src.data_loader import FinancialTimeSeriesLoader, generate_mock_financial_data

loader = FinancialTimeSeriesLoader(seq_len=24)
print("Successfully imported FinancialTimeSeriesLoader into Colab runtime!")